In [ ]:
import os, sys, glob, random, zipfile
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
from torch.utils.data import DataLoader, Dataset

REPO_DIR = Path("/kaggle/working/spiking-fullsubnet")
RECIPE_DIR = REPO_DIR / "recipes" / "intel_ndns" / "spiking_fullsubnet"

CKPT_URL = "https://huggingface.co/KhaBui/PESEM-VS/resolve/main/Spiking-Fullsubnet_EN.zip"
CKPT_ZIP = Path("/kaggle/working/Spiking-Fullsubnet_EN.zip")
CKPT_DIR = Path("/kaggle/working/Spiking-Fullsubnet_EN")

DATA_ROOT = Path("/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH")
TRAIN_CLEAN_DIR = DATA_ROOT / "TRAIN" / "CLEAN"
TRAIN_NOISE_DIR = DATA_ROOT / "TRAIN" / "NOISE"
TEST_DIR = DATA_ROOT / "TEST"

OUTPUT_DIR = Path("/kaggle/working/finetuned")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ENHANCED_DIR = Path("/kaggle/working/enhanced_test")
ENHANCED_DIR.mkdir(parents=True, exist_ok=True)

SR = 16000
TRAIN_CLIP_SECONDS = 6
VAL_FRACTION = 0.05
SNR_RANGE_DB = (-5, 15)
BATCH_SIZE = 8
NUM_EPOCHS = 20
LR = 1e-4
NUM_WORKERS = 2
SEED = 20220815

random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:
if not REPO_DIR.exists():
    os.system(f"git clone --depth 1 https://github.com/haoxiangsnr/spiking-fullsubnet.git {REPO_DIR}")

os.system(f"pip install -e {REPO_DIR} --quiet")
os.system("pip install simple_parsing einops safetensors --quiet")

sys.path.insert(0, str(RECIPE_DIR))  # model.py, efficient_spiking_neuron.py
sys.path.insert(0, str(REPO_DIR))    # audiozen package

from audiozen.loss import SISNRLoss, freq_MAE, mag_MAE
from model import ModelArgs, SpikingFullSubNet

print("Import OK")


In [ ]:
if not CKPT_ZIP.exists():
    os.system(f"wget -q -O '{CKPT_ZIP}' '{CKPT_URL}'")

if not CKPT_DIR.exists():
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(CKPT_ZIP, "r") as zf:
        zf.extractall(CKPT_DIR)

print("Checkpoint files:")
for p in CKPT_DIR.rglob("*"):
    print(" ", p)


In [ ]:
MODEL_ARGS_DICT = dict(
    n_fft=512,
    hop_length=128,
    win_length=512,
    fdrc=0.5,
    fb_input_size=64,
    fb_hidden_size=320,
    fb_num_layers=2,
    fb_proj_size=64,
    fb_output_activate_function=None,
    sb_hidden_size=224,
    sb_num_layers=2,
    freq_cutoffs=[0, 32, 128, 256],
    df_orders=[5, 3, 1],
    center_freq_sizes=[8, 32, 64],
    neighbor_freq_sizes=[15, 15, 15],
    use_pre_layer_norm_fb=True,
    use_pre_layer_norm_sb=True,
    bn=True,
    shared_weights=True,
    sequence_model="GSN",
    num_spks=1,
)

def build_model():
    args = ModelArgs(**MODEL_ARGS_DICT)
    return SpikingFullSubNet(args)


def find_state_dict_file(ckpt_dir: Path):
    candidates = []
    for pattern in ("**/*.safetensors", "**/pytorch_model.bin", "**/*.pt", "**/*.pth", "**/*.tar", "**/*.bin"):
        candidates += list(ckpt_dir.glob(pattern))
    if not candidates:
        raise FileNotFoundError(f"No weight file found under {ckpt_dir}. Inspect its contents manually.")
    candidates.sort(key=lambda p: (p.suffix != ".safetensors", -p.stat().st_size))
    return candidates[0]


def load_pretrained_weights(model, ckpt_dir: Path):
    weight_file = find_state_dict_file(ckpt_dir)
    print(f"Loading weights from: {weight_file}")

    if weight_file.suffix == ".safetensors":
        from safetensors.torch import load_file
        state_dict = load_file(str(weight_file))
    else:
        obj = torch.load(str(weight_file), map_location="cpu", weights_only=False)
        if isinstance(obj, dict) and "state_dict" in obj:
            state_dict = obj["state_dict"]
        elif isinstance(obj, dict) and "model" in obj and isinstance(obj["model"], dict):
            state_dict = obj["model"]
        else:
            state_dict = obj

    cleaned = {}
    for k, v in state_dict.items():
        nk = k
        for prefix in ("module.", "model.", "model_g."):
            if nk.startswith(prefix):
                nk = nk[len(prefix):]
        cleaned[nk] = v

    missing, unexpected = model.load_state_dict(cleaned, strict=False)
    print(f"Missing keys ({len(missing)}): {missing[:10]}{' ...' if len(missing) > 10 else ''}")
    print(f"Unexpected keys ({len(unexpected)}): {unexpected[:10]}{' ...' if len(unexpected) > 10 else ''}")
    if len(missing) > 5 or len(unexpected) > 5:
        print("*** WARNING: many mismatched keys -- MODEL_ARGS_DICT probably doesn't match this checkpoint. ***")
    return model


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = build_model()
model = load_pretrained_weights(model, CKPT_DIR)
model.to(device)
print("Model loaded.")


In [ ]:
def match_clean_noise_pairs(clean_dir: Path, noise_dir: Path):
    clean_files = sorted(glob.glob(str(clean_dir / "**" / "*.wav"), recursive=True))
    noise_by_stem = {Path(f).stem: f for f in glob.glob(str(noise_dir / "**" / "*.wav"), recursive=True)}

    pairs = []
    for cf in clean_files:
        nf = noise_by_stem.get(Path(cf).stem)
        if nf is not None:
            pairs.append((cf, nf))

    print(f"Matched {len(pairs)} pairs ({len(clean_files) - len(pairs)} clean files unmatched).")
    if not pairs:
        raise RuntimeError("No clean/noise pairs matched by filename -- check naming or adjust matching logic.")
    return pairs


def mix_at_snr(clean, noise, snr_db):
    if len(noise) < len(clean):
        noise = np.tile(noise, int(np.ceil(len(clean) / len(noise))))
    start = random.randint(0, max(0, len(noise) - len(clean)))
    noise = noise[start:start + len(clean)]

    clean_power = np.mean(clean**2) + 1e-12
    noise_power = np.mean(noise**2) + 1e-12
    scale = np.sqrt((clean_power / (10 ** (snr_db / 10))) / noise_power)
    return (clean + noise * scale).astype(np.float32)


class CleanNoisePairDataset(Dataset):
    def __init__(self, pairs, sr=SR, sublen=TRAIN_CLIP_SECONDS, snr_range=SNR_RANGE_DB, train=True):
        self.pairs = pairs
        self.sr = sr
        self.sublen_samples = sublen * sr
        self.snr_range = snr_range
        self.train = train

    def __len__(self):
        return len(self.pairs)

    def _load_mono(self, path):
        audio, sr = sf.read(path, dtype="float32")
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if sr != self.sr:
            import librosa
            audio = librosa.resample(audio, orig_sr=sr, target_sr=self.sr)
        return audio

    def __getitem__(self, idx):
        clean_path, noise_path = self.pairs[idx]
        clean = self._load_mono(clean_path)
        noise = self._load_mono(noise_path)
        noisy = mix_at_snr(clean, noise, random.uniform(*self.snr_range))

        n = self.sublen_samples
        if self.train:
            if len(clean) >= n:
                start = random.randint(0, len(clean) - n)
                clean, noisy = clean[start:start + n], noisy[start:start + n]
            else:
                pad = n - len(clean)
                clean = np.pad(clean, (0, pad))
                noisy = np.pad(noisy, (0, pad))
        return noisy.astype(np.float32), clean.astype(np.float32)


def collate_fixed_len(batch):
    noisy = torch.from_numpy(np.stack([b[0] for b in batch]))
    clean = torch.from_numpy(np.stack([b[1] for b in batch]))
    return noisy, clean


# ---- Speed: more workers, pin_memory, persistent_workers (fewer worker respawns) ----
NUM_WORKERS = 4  # bumped from 2 -- Kaggle usually gives 4 CPU cores

pairs = match_clean_noise_pairs(TRAIN_CLEAN_DIR, TRAIN_NOISE_DIR)
random.shuffle(pairs)
n_val = max(1, int(len(pairs) * VAL_FRACTION))
val_pairs, train_pairs = pairs[:n_val], pairs[n_val:]
print(f"Train pairs: {len(train_pairs)}, Val pairs: {len(val_pairs)}")

train_loader = DataLoader(
    CleanNoisePairDataset(train_pairs, train=True),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    collate_fn=collate_fixed_len, drop_last=True,
    pin_memory=True, persistent_workers=True, prefetch_factor=4,
)
val_loader = DataLoader(
    CleanNoisePairDataset(val_pairs, train=True),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    collate_fn=collate_fixed_len, drop_last=False,
    pin_memory=True, persistent_workers=True,
)


In [ ]:
torch.backends.cudnn.benchmark = True

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs via DataParallel")
    model = torch.nn.DataParallel(model)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
sisnr_loss_fn = SISNRLoss(return_neg=False)
best_val_loss = float("inf")

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running = torch.zeros(1, device=device)  # accumulate on GPU, avoid per-step sync
    for step, (noisy_y, clean_y) in enumerate(train_loader):
        noisy_y = noisy_y.to(device, non_blocking=True)
        clean_y = clean_y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        enhanced_y, *_ = model(noisy_y)
        loss_freq = freq_MAE(enhanced_y, clean_y)
        loss_mag = mag_MAE(enhanced_y, clean_y)
        loss_sdr = sisnr_loss_fn(enhanced_y, clean_y)
        loss_sdr_norm = 0.001 * (100 - loss_sdr)
        loss = loss_freq + loss_mag + loss_sdr_norm

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10)
        optimizer.step()

        running += loss.detach()
        if step % 50 == 0:
            print(f"Epoch {epoch} step {step}/{len(train_loader)} loss={loss.item():.4f}")

    train_loss = (running / max(1, len(train_loader))).item()

    model.eval()
    val_running = torch.zeros(1, device=device)
    with torch.no_grad():
        for noisy_y, clean_y in val_loader:
            noisy_y = noisy_y.to(device, non_blocking=True)
            clean_y = clean_y.to(device, non_blocking=True)
            enhanced_y, *_ = model(noisy_y)
            loss_freq = freq_MAE(enhanced_y, clean_y)
            loss_mag = mag_MAE(enhanced_y, clean_y)
            loss_sdr = sisnr_loss_fn(enhanced_y, clean_y)
            val_loss_step = loss_freq + loss_mag + 0.001 * (100 - loss_sdr)
            val_running += val_loss_step.detach()
    val_loss = (val_running / max(1, len(val_loader))).item()

    print(f"== Epoch {epoch}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} ==")

    # unwrap DataParallel before saving so the checkpoint loads back into a plain model
    state_dict = model.module.state_dict() if isinstance(model, torch.nn.DataParallel) else model.state_dict()

    torch.save({"model": state_dict, "epoch": epoch, "model_args": MODEL_ARGS_DICT},
               OUTPUT_DIR / f"epoch_{epoch:04d}.pt")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({"model": state_dict, "epoch": epoch, "model_args": MODEL_ARGS_DICT},
                   OUTPUT_DIR / "best.pt")
        print(f"  -> new best checkpoint saved (val_loss={val_loss:.4f})")


## Inference (enhance toan bo tap TEST bang checkpoint tot nhat vua finetune)

Load lai `best.pt` (checkpoint co val_loss thap nhat trong vong finetune o tren) vao mot model moi (khong boc trong `DataParallel`) roi chay forward tren tung file trong `TEST_DIR`, luu ket qua enhanced ra wav 16kHz.

In [ ]:
import glob
import soundfile as sf
import numpy as np
import librosa
from tqdm import tqdm

best_ckpt_path = OUTPUT_DIR / "best.pt"
assert best_ckpt_path.is_file(), f"Khong tim thay {best_ckpt_path}. Hay chay xong vong finetune o tren truoc."

best_ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
print(f"Load checkpoint tot nhat: epoch {best_ckpt.get('epoch')}")

eval_model = build_model()
eval_model.load_state_dict(best_ckpt["model"])
eval_model.to(device)
eval_model.eval()
print("Da load model cho inference.")


In [ ]:
ENHANCED_DIR = Path("/kaggle/working/enhanced_SpikingFullSubNet")
ENHANCED_DIR.mkdir(parents=True, exist_ok=True)

test_files = sorted(glob.glob(str(TEST_DIR / "*.wav")) + glob.glob(str(TEST_DIR / "**" / "*.wav"), recursive=True))
test_files = sorted(set(test_files))
print(f"So file test: {len(test_files)}")
assert len(test_files) > 0, f"Khong tim thay file .wav nao trong {TEST_DIR}"

with torch.no_grad():
    for fpath in tqdm(test_files):
        noisy, sr = sf.read(fpath, dtype="float32")
        if noisy.ndim > 1:
            noisy = noisy.mean(axis=1)
        if sr != SR:
            noisy = librosa.resample(noisy, orig_sr=sr, target_sr=SR)

        noisy_t = torch.from_numpy(noisy.astype(np.float32)).unsqueeze(0).to(device)
        enhanced_t, *_ = eval_model(noisy_t)
        enhanced = enhanced_t.squeeze(0).detach().cpu().numpy()

        out_path = ENHANCED_DIR / Path(fpath).name
        sf.write(str(out_path), enhanced, SR)

print("Da enhance xong toan bo tap test, luu tai:", ENHANCED_DIR)
print("Buoc tiep theo: chay volume.py de RMS-normalize ve -16 dBFS, roi metrics.py de tinh PESQ/STOI/F0-RMSE/PFR.")
